In [1]:
import duckdb
import polars as pl
import gc
from pathlib import Path

BASE_INTERACTIONS = Path("/kaggle/input/datasets/fedorfesenko/train-interactions")
WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

print(f"Input: {BASE_INTERACTIONS}")
print(f"Work:  {WORK}")

Input: /kaggle/input/datasets/fedorfesenko/train-interactions
Work:  /kaggle/working


In [2]:
df_interactions_train = pl.read_parquet(BASE_INTERACTIONS / "df_interactions_train.parquet")
df_interactions_val   = pl.read_parquet(BASE_INTERACTIONS / "df_interactions_val.parquet")
df_ranking_train      = pl.read_parquet(BASE_INTERACTIONS / "df_ranking_train.parquet")
df_ranking_val        = pl.read_parquet(BASE_INTERACTIONS / "df_ranking_val.parquet")
df_test               = pl.read_parquet(BASE_INTERACTIONS / "df_test.parquet")

print(f"interactions_train: {df_interactions_train.shape}")
print(f"interactions_val:   {df_interactions_val.shape}")
print(f"ranking_train:      {df_ranking_train.shape}")
print(f"ranking_val:        {df_ranking_val.shape}")
print(f"test:               {df_test.shape}")

interactions_train: (16175519, 5)
interactions_val:   (333095, 5)
ranking_train:      (15769472, 14)
ranking_val:        (323736, 14)
test:               (292826, 11)


In [3]:
T_TRAIN_END = df_interactions_train["timestamp"].max()
T_VAL_END   = df_interactions_val["timestamp"].max()

print(f"T_TRAIN_END = {T_TRAIN_END}")
print(f"T_VAL_END   = {T_VAL_END}")

# actions = train + val события
actions = pl.concat([
    df_interactions_train.select(["user_id", "product_id", "action_type", "timestamp", "product_category"]),
    df_interactions_val.select(["user_id", "product_id", "action_type", "timestamp", "product_category"]),
])
print(f"actions: {actions.shape}")

actions.write_parquet(WORK / "actions.parquet")
print("Saved actions.parquet")

del actions
gc.collect()

T_TRAIN_END = 1706290114
T_VAL_END   = 1706895291
actions: (16508614, 5)
Saved actions.parquet


20

In [4]:
df_ranking_train_ft = df_ranking_train.select(
    ["user_id", "product_id", "request_id", "timestamp"]
).with_columns(
    pl.lit("train").alias("split"),
    pl.col("timestamp").alias("feature_timestamp"),
)

df_ranking_val_ft = df_ranking_val.select(
    ["user_id", "product_id", "request_id", "timestamp"]
).with_columns(
    pl.lit("val").alias("split"),
    pl.lit(T_TRAIN_END, dtype=pl.Int64).alias("feature_timestamp"),
)

df_test_ft = df_test.select(
    ["user_id", "product_id", "request_id", "timestamp"]
).with_columns(
    pl.lit("test").alias("split"),
    pl.lit(T_VAL_END, dtype=pl.Int64).alias("feature_timestamp"),
)

df_features_rows = pl.concat([df_ranking_train_ft, df_ranking_val_ft, df_test_ft])
print(f"df_features_rows: {df_features_rows.shape}")

df_features_rows.write_parquet(WORK / "df_features_rows.parquet")
print("Saved df_features_rows.parquet")

del df_ranking_train_ft, df_ranking_val_ft, df_test_ft, df_features_rows
gc.collect()

df_features_rows: (16386034, 6)
Saved df_features_rows.parquet


0

In [5]:
duckdb.sql(f"""
    COPY (
        WITH base AS (
            SELECT
                user_id,
                product_id,
                product_category,
                action_type,
                timestamp,
                date_trunc('day', make_timestamp(timestamp * 1000000)) AS day
            FROM '{WORK}/actions.parquet'
        ),
        windows AS (
            SELECT
                user_id,
                timestamp,
                day,
                coalesce(sum(CASE WHEN action_type = 'AT_View'       THEN 1 ELSE 0 END) OVER rolling_all, 0) AS n_views_all,
                coalesce(sum(CASE WHEN action_type = 'AT_Click'      THEN 1 ELSE 0 END) OVER rolling_all, 0) AS n_clicks_all,
                coalesce(sum(CASE WHEN action_type = 'AT_CartUpdate' THEN 1 ELSE 0 END) OVER rolling_all, 0) AS n_cart_updates_all,
                coalesce(sum(CASE WHEN action_type = 'AT_Purchase'   THEN 1 ELSE 0 END) OVER rolling_all, 0) AS n_purchases_all,
                coalesce(sum(CASE WHEN action_type = 'AT_View'       THEN 1 ELSE 0 END) OVER rolling_30d, 0) AS n_views_30d,
                coalesce(sum(CASE WHEN action_type = 'AT_Click'      THEN 1 ELSE 0 END) OVER rolling_30d, 0) AS n_clicks_30d,
                coalesce(sum(CASE WHEN action_type = 'AT_CartUpdate' THEN 1 ELSE 0 END) OVER rolling_30d, 0) AS n_cart_updates_30d,
                coalesce(sum(CASE WHEN action_type = 'AT_Purchase'   THEN 1 ELSE 0 END) OVER rolling_30d, 0) AS n_purchases_30d,
                coalesce(sum(CASE WHEN action_type = 'AT_View'       THEN 1 ELSE 0 END) OVER rolling_7d, 0)  AS n_views_7d,
                coalesce(sum(CASE WHEN action_type = 'AT_Click'      THEN 1 ELSE 0 END) OVER rolling_7d, 0)  AS n_clicks_7d,
                coalesce(sum(CASE WHEN action_type = 'AT_CartUpdate' THEN 1 ELSE 0 END) OVER rolling_7d, 0)  AS n_cart_updates_7d,
                coalesce(sum(CASE WHEN action_type = 'AT_Purchase'   THEN 1 ELSE 0 END) OVER rolling_7d, 0)  AS n_purchases_7d,
                coalesce(sum(CASE WHEN action_type = 'AT_View'       THEN 1 ELSE 0 END) OVER rolling_1d, 0)  AS n_views_1d,
                coalesce(sum(CASE WHEN action_type = 'AT_Click'      THEN 1 ELSE 0 END) OVER rolling_1d, 0)  AS n_clicks_1d,
                coalesce(sum(CASE WHEN action_type = 'AT_CartUpdate' THEN 1 ELSE 0 END) OVER rolling_1d, 0)  AS n_cart_updates_1d,
                coalesce(sum(CASE WHEN action_type = 'AT_Purchase'   THEN 1 ELSE 0 END) OVER rolling_1d, 0)  AS n_purchases_1d,
                min(timestamp) OVER rolling_all AS first_action_ts,
                max(timestamp) OVER rolling_all AS last_action_ts
            FROM base
            WINDOW
                rolling_all AS (PARTITION BY user_id ORDER BY day ASC
                                RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_30d AS (PARTITION BY user_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_7d  AS (PARTITION BY user_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 7 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_1d  AS (PARTITION BY user_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 1 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING)
        )
        SELECT
            user_id,
            timestamp AS ts,
            n_views_all, n_clicks_all, n_cart_updates_all, n_purchases_all,
            n_views_30d, n_clicks_30d, n_cart_updates_30d, n_purchases_30d,
            n_views_7d,  n_clicks_7d,  n_cart_updates_7d,  n_purchases_7d,
            n_views_1d,  n_clicks_1d,  n_cart_updates_1d,  n_purchases_1d,
            coalesce(datepart('day', make_timestamp(timestamp * 1000000) - make_timestamp(first_action_ts * 1000000)), -1) AS days_since_first_action,
            coalesce(datepart('day', make_timestamp(timestamp * 1000000) - make_timestamp(last_action_ts * 1000000)),  -1) AS days_since_last_action
        FROM windows
    ) TO '{WORK}/user_counters_raw.parquet' (FORMAT PARQUET)
""")

print("Saved user_counters_raw.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved user_counters_raw.parquet


In [6]:
duckdb.sql(f"""
    COPY (
        WITH base AS (
            SELECT
                user_id,
                product_id,
                product_category,
                timestamp,
                date_trunc('day', make_timestamp(timestamp * 1000000)) AS day
            FROM '{WORK}/actions.parquet'
        ),
        items_daily AS (
            SELECT user_id, day, count(DISTINCT product_id) AS n_unique_items_day
            FROM base
            GROUP BY user_id, day
        ),
        cats_daily AS (
            SELECT user_id, day, count(DISTINCT product_category) AS n_unique_categories_day
            FROM base
            GROUP BY user_id, day
        ),
        items_windows AS (
            SELECT
                user_id, day,
                coalesce(sum(n_unique_items_day) OVER rolling_all, 0) AS n_unique_items_all,
                coalesce(sum(n_unique_items_day) OVER rolling_30d, 0) AS n_unique_items_30d,
                coalesce(sum(n_unique_items_day) OVER rolling_7d,  0) AS n_unique_items_7d,
                coalesce(sum(n_unique_items_day) OVER rolling_1d,  0) AS n_unique_items_1d
            FROM items_daily
            WINDOW
                rolling_all AS (PARTITION BY user_id ORDER BY day ASC
                                RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_30d AS (PARTITION BY user_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_7d  AS (PARTITION BY user_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 7 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_1d  AS (PARTITION BY user_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 1 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING)
        ),
        cats_windows AS (
            SELECT
                user_id, day,
                coalesce(sum(n_unique_categories_day) OVER rolling_all, 0) AS n_unique_categories_all,
                coalesce(sum(n_unique_categories_day) OVER rolling_30d, 0) AS n_unique_categories_30d,
                coalesce(sum(n_unique_categories_day) OVER rolling_7d,  0) AS n_unique_categories_7d,
                coalesce(sum(n_unique_categories_day) OVER rolling_1d,  0) AS n_unique_categories_1d
            FROM cats_daily
            WINDOW
                rolling_all AS (PARTITION BY user_id ORDER BY day ASC
                                RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_30d AS (PARTITION BY user_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_7d  AS (PARTITION BY user_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 7 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_1d  AS (PARTITION BY user_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 1 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING)
        )
        SELECT
            i.user_id, i.day,
            i.n_unique_items_all, i.n_unique_items_30d, i.n_unique_items_7d, i.n_unique_items_1d,
            c.n_unique_categories_all, c.n_unique_categories_30d, c.n_unique_categories_7d, c.n_unique_categories_1d
        FROM items_windows i
        JOIN cats_windows c ON i.user_id = c.user_id AND i.day = c.day
    ) TO '{WORK}/user_unique_counters_raw.parquet' (FORMAT PARQUET)
""")

print("Saved user_unique_counters_raw.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved user_unique_counters_raw.parquet


In [7]:
duckdb.sql(f"""
    COPY (
        WITH base AS (
            SELECT
                product_id,
                user_id,
                action_type,
                timestamp,
                date_trunc('day', make_timestamp(timestamp * 1000000)) AS day
            FROM '{WORK}/actions.parquet'
        ),
        windows AS (
            SELECT
                product_id, timestamp, day,
                coalesce(sum(CASE WHEN action_type = 'AT_View'       THEN 1 ELSE 0 END) OVER rolling_all, 0) AS item_n_views_all,
                coalesce(sum(CASE WHEN action_type = 'AT_Click'      THEN 1 ELSE 0 END) OVER rolling_all, 0) AS item_n_clicks_all,
                coalesce(sum(CASE WHEN action_type = 'AT_CartUpdate' THEN 1 ELSE 0 END) OVER rolling_all, 0) AS item_n_cart_updates_all,
                coalesce(sum(CASE WHEN action_type = 'AT_Purchase'   THEN 1 ELSE 0 END) OVER rolling_all, 0) AS item_n_purchases_all,
                coalesce(sum(CASE WHEN action_type = 'AT_View'       THEN 1 ELSE 0 END) OVER rolling_30d, 0) AS item_n_views_30d,
                coalesce(sum(CASE WHEN action_type = 'AT_Click'      THEN 1 ELSE 0 END) OVER rolling_30d, 0) AS item_n_clicks_30d,
                coalesce(sum(CASE WHEN action_type = 'AT_CartUpdate' THEN 1 ELSE 0 END) OVER rolling_30d, 0) AS item_n_cart_updates_30d,
                coalesce(sum(CASE WHEN action_type = 'AT_Purchase'   THEN 1 ELSE 0 END) OVER rolling_30d, 0) AS item_n_purchases_30d,
                coalesce(sum(CASE WHEN action_type = 'AT_View'       THEN 1 ELSE 0 END) OVER rolling_7d, 0)  AS item_n_views_7d,
                coalesce(sum(CASE WHEN action_type = 'AT_Click'      THEN 1 ELSE 0 END) OVER rolling_7d, 0)  AS item_n_clicks_7d,
                coalesce(sum(CASE WHEN action_type = 'AT_CartUpdate' THEN 1 ELSE 0 END) OVER rolling_7d, 0)  AS item_n_cart_updates_7d,
                coalesce(sum(CASE WHEN action_type = 'AT_Purchase'   THEN 1 ELSE 0 END) OVER rolling_7d, 0)  AS item_n_purchases_7d,
                coalesce(sum(CASE WHEN action_type = 'AT_View'       THEN 1 ELSE 0 END) OVER rolling_1d, 0)  AS item_n_views_1d,
                coalesce(sum(CASE WHEN action_type = 'AT_Click'      THEN 1 ELSE 0 END) OVER rolling_1d, 0)  AS item_n_clicks_1d,
                coalesce(sum(CASE WHEN action_type = 'AT_CartUpdate' THEN 1 ELSE 0 END) OVER rolling_1d, 0)  AS item_n_cart_updates_1d,
                coalesce(sum(CASE WHEN action_type = 'AT_Purchase'   THEN 1 ELSE 0 END) OVER rolling_1d, 0)  AS item_n_purchases_1d,
                min(timestamp) OVER rolling_all AS first_view_ts,
                max(timestamp) OVER rolling_all AS last_view_ts
            FROM base
            WINDOW
                rolling_all AS (PARTITION BY product_id ORDER BY day ASC
                                RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_30d AS (PARTITION BY product_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_7d  AS (PARTITION BY product_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 7 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_1d  AS (PARTITION BY product_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 1 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING)
        )
        SELECT
            product_id, timestamp AS ts,
            item_n_views_all, item_n_clicks_all, item_n_cart_updates_all, item_n_purchases_all,
            item_n_views_30d, item_n_clicks_30d, item_n_cart_updates_30d, item_n_purchases_30d,
            item_n_views_7d,  item_n_clicks_7d,  item_n_cart_updates_7d,  item_n_purchases_7d,
            item_n_views_1d,  item_n_clicks_1d,  item_n_cart_updates_1d,  item_n_purchases_1d,
            coalesce(datepart('day', make_timestamp(timestamp * 1000000) - make_timestamp(first_view_ts * 1000000)), -1) AS item_days_since_first_view,
            coalesce(datepart('day', make_timestamp(timestamp * 1000000) - make_timestamp(last_view_ts * 1000000)),  -1) AS item_days_since_last_view
        FROM windows
    ) TO '{WORK}/item_counters_raw.parquet' (FORMAT PARQUET)
""")

print("Saved item_counters_raw.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved item_counters_raw.parquet


In [8]:
duckdb.sql(f"""
    COPY (
        WITH base AS (
            SELECT
                product_id, user_id, timestamp,
                date_trunc('day', make_timestamp(timestamp * 1000000)) AS day
            FROM '{WORK}/actions.parquet'
        ),
        users_daily AS (
            SELECT product_id, day, count(DISTINCT user_id) AS n_unique_users_day
            FROM base
            GROUP BY product_id, day
        ),
        windows AS (
            SELECT
                product_id, day,
                coalesce(sum(n_unique_users_day) OVER rolling_all, 0) AS item_n_unique_users_all,
                coalesce(sum(n_unique_users_day) OVER rolling_30d, 0) AS item_n_unique_users_30d,
                coalesce(sum(n_unique_users_day) OVER rolling_7d,  0) AS item_n_unique_users_7d,
                coalesce(sum(n_unique_users_day) OVER rolling_1d,  0) AS item_n_unique_users_1d
            FROM users_daily
            WINDOW
                rolling_all AS (PARTITION BY product_id ORDER BY day ASC
                                RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_30d AS (PARTITION BY product_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_7d  AS (PARTITION BY product_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 7 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_1d  AS (PARTITION BY product_id ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 1 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING)
        )
        SELECT * FROM windows
    ) TO '{WORK}/item_unique_counters_raw.parquet' (FORMAT PARQUET)
""")

print("Saved item_unique_counters_raw.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved item_unique_counters_raw.parquet


In [9]:
duckdb.sql(f"""
    COPY (
        WITH base AS (
            SELECT
                user_id, product_id, action_type, timestamp,
                date_trunc('day', make_timestamp(timestamp * 1000000)) AS day
            FROM '{WORK}/actions.parquet'
        ),
        windows AS (
            SELECT
                user_id, product_id, timestamp, day,
                coalesce(sum(CASE WHEN action_type = 'AT_View'       THEN 1 ELSE 0 END) OVER rolling_all, 0) AS ui_n_views_all,
                coalesce(sum(CASE WHEN action_type = 'AT_Click'      THEN 1 ELSE 0 END) OVER rolling_all, 0) AS ui_n_clicks_all,
                coalesce(sum(CASE WHEN action_type = 'AT_CartUpdate' THEN 1 ELSE 0 END) OVER rolling_all, 0) AS ui_n_cart_updates_all,
                coalesce(sum(CASE WHEN action_type = 'AT_Purchase'   THEN 1 ELSE 0 END) OVER rolling_all, 0) AS ui_n_purchases_all,
                max(CASE WHEN action_type = 'AT_CartUpdate' THEN timestamp END) OVER rolling_all AS ui_last_cart_update_ts,
                max(CASE WHEN action_type = 'AT_Purchase'   THEN timestamp END) OVER rolling_all AS ui_last_purchase_ts,
                max(timestamp) OVER rolling_all AS ui_last_action_ts
            FROM base
            WINDOW
                rolling_all AS (PARTITION BY user_id, product_id ORDER BY day ASC
                                RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL 1 DAYS PRECEDING)
        )
        SELECT
            user_id, product_id, timestamp AS ts,
            ui_n_views_all, ui_n_clicks_all, ui_n_cart_updates_all, ui_n_purchases_all,
            coalesce(datepart('day', make_timestamp(timestamp * 1000000) - make_timestamp(ui_last_action_ts * 1000000)),      -1) AS ui_days_since_last_action,
            coalesce(datepart('day', make_timestamp(timestamp * 1000000) - make_timestamp(ui_last_cart_update_ts * 1000000)), -1) AS ui_days_since_last_cart_update,
            coalesce(datepart('day', make_timestamp(timestamp * 1000000) - make_timestamp(ui_last_purchase_ts * 1000000)),    -1) AS ui_days_since_last_purchase
        FROM windows
    ) TO '{WORK}/ui_counters_raw.parquet' (FORMAT PARQUET)
""")

print("Saved ui_counters_raw.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved ui_counters_raw.parquet


In [10]:
duckdb.sql(f"""
    COPY (
        WITH base AS (
            SELECT
                user_id, product_category, action_type, timestamp,
                date_trunc('day', make_timestamp(timestamp * 1000000)) AS day
            FROM '{WORK}/actions.parquet'
        ),
        windows AS (
            SELECT
                user_id, product_category, timestamp, day,
                coalesce(sum(CASE WHEN action_type = 'AT_View'       THEN 1 ELSE 0 END) OVER rolling_all, 0) AS uc_n_views_all,
                coalesce(sum(CASE WHEN action_type = 'AT_Click'      THEN 1 ELSE 0 END) OVER rolling_all, 0) AS uc_n_clicks_all,
                coalesce(sum(CASE WHEN action_type = 'AT_CartUpdate' THEN 1 ELSE 0 END) OVER rolling_all, 0) AS uc_n_cart_updates_all,
                coalesce(sum(CASE WHEN action_type = 'AT_Purchase'   THEN 1 ELSE 0 END) OVER rolling_all, 0) AS uc_n_purchases_all,
                coalesce(sum(CASE WHEN action_type = 'AT_View'       THEN 1 ELSE 0 END) OVER rolling_30d, 0) AS uc_n_views_30d,
                coalesce(sum(CASE WHEN action_type = 'AT_Click'      THEN 1 ELSE 0 END) OVER rolling_30d, 0) AS uc_n_clicks_30d,
                coalesce(sum(CASE WHEN action_type = 'AT_CartUpdate' THEN 1 ELSE 0 END) OVER rolling_30d, 0) AS uc_n_cart_updates_30d,
                coalesce(sum(CASE WHEN action_type = 'AT_Purchase'   THEN 1 ELSE 0 END) OVER rolling_30d, 0) AS uc_n_purchases_30d,
                coalesce(sum(CASE WHEN action_type = 'AT_View'       THEN 1 ELSE 0 END) OVER rolling_7d, 0)  AS uc_n_views_7d,
                coalesce(sum(CASE WHEN action_type = 'AT_Click'      THEN 1 ELSE 0 END) OVER rolling_7d, 0)  AS uc_n_clicks_7d,
                coalesce(sum(CASE WHEN action_type = 'AT_CartUpdate' THEN 1 ELSE 0 END) OVER rolling_7d, 0)  AS uc_n_cart_updates_7d,
                coalesce(sum(CASE WHEN action_type = 'AT_Purchase'   THEN 1 ELSE 0 END) OVER rolling_7d, 0)  AS uc_n_purchases_7d,
                max(CASE WHEN action_type = 'AT_Purchase' THEN timestamp END) OVER rolling_all AS uc_last_purchase_ts
            FROM base
            WINDOW
                rolling_all AS (PARTITION BY user_id, product_category ORDER BY day ASC
                                RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_30d AS (PARTITION BY user_id, product_category ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING),
                rolling_7d  AS (PARTITION BY user_id, product_category ORDER BY day ASC
                                RANGE BETWEEN INTERVAL 7 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING)
        )
        SELECT
            user_id, product_category, timestamp AS ts,
            uc_n_views_all, uc_n_clicks_all, uc_n_cart_updates_all, uc_n_purchases_all,
            uc_n_views_30d, uc_n_clicks_30d, uc_n_cart_updates_30d, uc_n_purchases_30d,
            uc_n_views_7d,  uc_n_clicks_7d,  uc_n_cart_updates_7d,  uc_n_purchases_7d,
            coalesce(datepart('day', make_timestamp(timestamp * 1000000) - make_timestamp(uc_last_purchase_ts * 1000000)), -1) AS uc_days_since_last_purchase
        FROM windows
    ) TO '{WORK}/uc_counters_raw.parquet' (FORMAT PARQUET)
""")

print("Saved uc_counters_raw.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved uc_counters_raw.parquet


In [11]:
!ls -lh /kaggle/working/*.parquet

-rw-r--r-- 1 root root 152M Oct 16 07:18 /kaggle/working/actions.parquet
-rw-r--r-- 1 root root 297M Oct 16 07:19 /kaggle/working/df_features_rows.parquet
-rw-r--r-- 1 root root 141M Oct 16 07:27 /kaggle/working/item_counters_raw.parquet
-rw-r--r-- 1 root root  15M Oct 16 07:27 /kaggle/working/item_unique_counters_raw.parquet
-rw-r--r-- 1 root root 105M Oct 16 07:28 /kaggle/working/uc_counters_raw.parquet
-rw-r--r-- 1 root root 148M Oct 16 07:27 /kaggle/working/ui_counters_raw.parquet
-rw-r--r-- 1 root root  38M Oct 16 07:25 /kaggle/working/user_counters_raw.parquet
-rw-r--r-- 1 root root 1.1M Oct 16 07:25 /kaggle/working/user_unique_counters_raw.parquet
